# --- Model Orchestrator ---


In [1]:
# ==========================
# Libraries & Custom Scripts
# ==========================
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import torch


In [2]:
# =============================
# Create Workspace directory
# =============================
PROJECT_ROOT = Path(os.getcwd()).parent

# Connect to custom-defined modules
sys.path.append(str(PROJECT_ROOT))

# Auto-reload scripts if any changes
%reload_ext autoreload
%autoreload 2

print("Workspace successfully configured.")

Workspace successfully configured.


In [3]:
# =========================
# Import custom Modules
# =========================
from src.data.dataloader import create_multitask_dataloader

from src.models.config import MultiTaskModelConfig
from src.models.network import MultiTaskYOLO
from src.models.loss import MultiTaskUncertaintyLoss

from src.training.trainer import MultiTaskTrainer

## Step 1: Verify model initialization, layer channel shapes, and multi-head routing logic.

In [5]:
# ======================================
# Test Multi-Task Network Architecture
# ======================================

# 1. Config & Model
print("⚙️ Initializing Multi-Task Network Model Architecture...")
config = MultiTaskModelConfig()
# Notice the parentheses () added to config.weights_path()
weights_file = config.weights_path() if callable(config.weights_path) else config.weights_path

# Verify file existence before initialization
if os.path.exists(config.weights_path):
    print("✅ Weights file located successfully!")
else:
    print(f"⚠️ Warning: File not found at {config.weights_path}. Ultralytics will auto-download it there.")

model = MultiTaskYOLO(config)
print("✅ MultiTaskYOLO successfully instantiated with weights from isolated folder!")

# Dummy batch input simulating DataLoader outputs
dummy_images = torch.randn(8, 3, 640, 640)
dummy_task_ids = torch.tensor([0, 0, 1, 1, 2, 2, 0, 1])  # Mixed stream tasks

print("\n📋 ================= MODEL CONFIGURATION REPORT =================")
print(f"• Configured Backbone : {config.backbone_type.upper()}")
print(f"• Active Task Heads   : {list(config.heads.keys())}")
for name, cfg in config.heads.items():
    print(f"- [{name.upper()}] Classes: {cfg.num_classes} | Loss Weight: {cfg.loss_weight}")

# Test forward pass
model.eval()
with torch.no_grad():
    outputs = model(dummy_images, dummy_task_ids)

print("✅ Model forward pass completed successfully without argument mismatch errors!")

print("\n✅ SUCCESS: Network Multi-Head Routing Verification Complete!")
for task_name, res in outputs.items():
    print(f"• Head '{task_name}' processed {res['mask'].sum().item()} samples from batch.")

⚙️ Initializing Multi-Task Network Model Architecture...
✅ Weights file located successfully!
✅ MultiTaskYOLO successfully instantiated with weights from isolated folder!

📋 ================= MODEL CONFIGURATION REPORT =================
• Configured Backbone : YOLOV8M
• Active Task Heads   : ['turnaround', 'ppe', 'fod']
- [TURNAROUND] Classes: 13 | Loss Weight: 1.0
- [PPE] Classes: 2 | Loss Weight: 1.2
- [FOD] Classes: 1 | Loss Weight: 1.5


✅ Model forward pass completed successfully without argument mismatch errors!

✅ SUCCESS: Network Multi-Head Routing Verification Complete!
• Head 'turnaround' processed 3 samples from batch.
• Head 'ppe' processed 3 samples from batch.
• Head 'fod' processed 2 samples from batch.


In [6]:
# ================================
# Verify: Loss computation,
# Uncertainty parameter gradients,
# & logging dict keys
# ================================
loss_fn = MultiTaskUncertaintyLoss(model, config)

print("✅ SUCCESS: Multi-Task Uncertainty Loss Module initialized cleanly!")
print("\n📋 Learnable Task Uncertainty Parameters:")
for task, param in loss_fn.log_vars.items():
    print(f"• Head '{task}': log_var = {param.item():.4f} | initial sigma = {torch.exp(0.5 * param).item():.4f}")

print("\n✅ Multi-Task Uncertainty Loss Module initialized successfully!")

✅ SUCCESS: Multi-Task Uncertainty Loss Module initialized cleanly!

📋 Learnable Task Uncertainty Parameters:
• Head 'fod': log_var = 0.0000 | initial sigma = 1.0000
• Head 'ppe': log_var = 0.0000 | initial sigma = 1.0000
• Head 'turnaround': log_var = 0.0000 | initial sigma = 1.0000

✅ Multi-Task Uncertainty Loss Module initialized successfully!


## Step 2: Perform a 1-Epoch Dry Run on Actual Dataset

In [4]:
# =================================
# Run DataLoader Verification Cell
# =================================

# Configure paths to the split dataset directories
data_config = {
    "turnaround": {
        "images": str(PROJECT_ROOT / "data/processed/train/turnaround/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/turnaround/labels")
    },
    "ppe": {
        "images": str(PROJECT_ROOT / "data/processed/train/ppe/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/ppe/labels")
    },
    "fod": {
        "images": str(PROJECT_ROOT / "data/processed/train/fod/images"),
        "labels": str(PROJECT_ROOT / "data/processed/train/fod/labels")
    }
}

train_loader = create_multitask_dataloader(
    data_dirs=data_config,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

# Fetch one batch to verify
for images, task_targets, task_ids in train_loader:
    print("✅ DataLoader Batch Ready!")
    print(f"• Images Tensor: {images.shape}")
    print(f"• Task IDs: {task_ids.tolist()}")
    break

📦 Loaded [TURNAROUND] dataset: 6813 samples
📦 Loaded [PPE] dataset: 5641 samples
📦 Loaded [FOD] dataset: 23655 samples
✅ Combined Multi-Task Dataset total size: 36109 samples

✅ DataLoader Batch Ready!
• Images Tensor: torch.Size([8, 3, 640, 640])
• Task IDs: [2, 0, 2, 2, 2, 1, 2, 1]


In [7]:
# ==========================
# Run a 1-Epoch Dry Run
# ==========================

# 2. Optimizer & Scheduler
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

# 3. Trainer Instance
trainer = MultiTaskTrainer(
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=train_loader, # Pass train_loader temporarily for sanity check
    optimizer=optimizer,
    config=config,
    use_wandb=False,
    exp_name="dry_run_test"
)

# 4. Launch 1 Dry-Run Epoch
trainer.fit(num_epochs=1)

🚀 Starting Multi-Task Model Training on device [CUDA] for 1 Epcohs...



Epoch 1 [Train]:   0%|          | 0/4514 [00:00<?, ?it/s]

Epoch [01/01] | Train Loss: 134.8212 | Val Loss: 23.2919
 💾 Saved Best Model Checkpoint (Val LOss: 23.2919) -> /teamspace/studios/this_studio/aiport-incident-detection-final/model/checkpoints/best_multitask_model.pt

✅ Multi-Task Training Pipeline Completed Successfully!
